In [6]:
import requests
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from umap import UMAP
import pandas as pd
import numpy as np
from codes.utils import Config
import plotly.express as px

def reqEmbedding(text, embedding):

    response = requests.post(
                url="https://litellm.toxpipe.niehs.nih.gov/embeddings",
                headers={
                    "Authorization": f"Bearer {Config.env_config['OPENAI_API_KEY']}",
                    "Content-Type": "application/json"},
                json={
                    "input": text,
                    "model": embedding,
                    "encoding_format": "float"}
    )

    if response.ok: return response.json()['data'][0]['embedding']
    print(response.text)

In [11]:
df_embed = pd.DataFrame({x: reqEmbedding(x, 'text-embedding-ada-002') for x in ['hello', 'helloooo']})#, 'world', 'hi', 'globe', 'map']})
df_embed

,hello,helloooo
0,-0.025141,-0.022019
1,-0.019467,0.000538
2,-0.028037,-0.018638
3,-0.031026,-0.030095
4,-0.024718,-0.036119
...,...,...
1531,0.029201,0.007741
1532,0.002374,-0.001929
1533,-0.016439,-0.018138
1534,-0.005055,0.004472


In [12]:
RANDOM_STATE = 1000
clustering = 'kmeans'
dbscan_eps = 0.5
dbscan_min_samples = 2
kmeans_n_clusters = 2
tsne_perplexity = 2

In [14]:
scaler = StandardScaler()
X_t = scaler.fit_transform(df_embed.values).T

if clustering == 'dbscan':
    X_c = DBSCAN(eps=dbscan_eps, min_samples=dbscan_min_samples).fit(X_t)
else:
    X_c = KMeans(n_clusters=kmeans_n_clusters, max_iter=3000, random_state=RANDOM_STATE, verbose=True).fit(X_t)

#X_tsne = TSNE(n_components=2, perplexity=tsne_perplexity, max_iter=1000, random_state=RANDOM_STATE).fit_transform(X_t)
X_umap = UMAP().fit_transform(X_t)

df_plot = pd.DataFrame(X_umap, columns=['X', 'Y'])
df_plot['Cluster'] = list(map(str, X_c.labels_))
df_plot['Model'] = df_embed.columns

fig = px.scatter(df_plot, 
                x='X', 
                y='Y', 
                color='Cluster',
                symbol='Model',
                opacity=0.5,
                hover_name='Model',
                template='simple_white'
                )
fig.update_traces(marker={'size': 15})

fig.show()

Initialization complete
Iteration 0, inertia 0.0.
Converged at iteration 0: center shift 0.0 within tolerance 4.419487325443414e-06.


/Users/talukdera2/Desktop/projects/ods/ToxPipe-Model-Comparison/.venv/lib/python3.12/site-packages/umap/umap_.py:2462: UserWarning:

n_neighbors is larger than the dataset size; truncating to X.shape[0] - 1

/Users/talukdera2/Desktop/projects/ods/ToxPipe-Model-Comparison/.venv/lib/python3.12/site-packages/umap/umap_.py:134: UserWarning:

A large number of your vertices were disconnected from the manifold.
Disconnection_distance = inf has removed 0 edges.
It has fully disconnected 2 vertices.
You might consider using find_disconnected_points() to find and remove these points from your data.
Use umap.utils.disconnected_vertices() to identify them.



ValueError: zero-size array to reduction operation maximum which has no identity

In [26]:
scaler = StandardScaler()
X_t = scaler.fit_transform(X.values).T
X_t

array([[-0.9556291 , -0.73317113, -1.06919163, ..., -0.61442309,
        -0.16808101, -0.20412024],
       [-0.83480383,  0.04961332, -0.70226532, ..., -0.68266819,
         0.203844  , -0.9085509 ],
       [ 0.51961251, -0.70773524, -0.03590634, ..., -1.27592753,
        -0.6363202 , -0.79791623],
       [-1.18332483, -0.76964633, -0.73351945, ..., -0.47274431,
        -0.01861389,  0.2960972 ],
       [-0.62254511, -0.58575968,  0.51911675, ..., -0.77809494,
        -0.19071054, -1.09707702],
       [-0.8548493 , -0.83084655, -0.17477274, ..., -0.52766961,
        -0.71369054, -1.12402243]])

In [27]:
X_c = KMeans(n_clusters=2, max_iter=1000, random_state=Config.RANDOM_STATE, verbose=True).fit(X_t)
X_c.labels_

Initialization complete
Iteration 0, inertia 1690.0378039369741.
Iteration 1, inertia 791.3009417369431.
Converged at iteration 1: strict convergence.


array([0, 0, 1, 0, 1, 1], dtype=int32)

In [28]:
TSNE(n_components=2, learning_rate='auto', random_state=Config.RANDOM_STATE, perplexity=1).fit_transform(X_t)

array([[-172.3698   ,  -42.242733 ],
       [-175.08197  ,  -54.78695  ],
       [ 363.94705  ,   -4.7924614],
       [-170.58762  ,  -33.996796 ],
       [ 363.63174  ,  -13.222839 ],
       [ 364.42816  ,    8.032484 ]], dtype=float32)

In [9]:
import plotly.express as px

px.data.medals_wide(indexed=True)

medal,gold,silver,bronze
nation,,,
South Korea,24,13,11
China,10,15,8
Canada,9,12,12
